# 04 Customer Segmentation And Retention

Customer-level metrics help separate valuable repeat behaviour from expensive repeat behaviour.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/processed/food_delivery_cleaned.csv')

In [ ]:
customer = df.groupby('customer_id').agg(orders=('order_id','count'), total_order_value=('order_value','sum'), total_profit=('gross_profit','sum'), avg_margin=('profit_margin_pct','mean'), avg_discount=('discount_amount','mean'), delay_rate=('delivery_delay_flag','mean'), avg_rating=('rating','mean'), repeat_rate=('repeat_customer_flag','mean')).reset_index()
customer['profit_quartile'] = pd.qcut(customer['total_profit'], 4, labels=[1,2,3,4])
customer['frequency_quartile'] = pd.qcut(customer['orders'].rank(method='first'), 4, labels=[1,2,3,4])
def segment(row):
    if row['profit_quartile'] == 4 and row['frequency_quartile'] >= 3: return 'High-value loyal'
    if row['profit_quartile'] <= 2 and row['frequency_quartile'] == 4: return 'High-volume low-margin'
    if row['profit_quartile'] == 4: return 'High-profit occasional'
    return 'Standard'
customer['customer_segment'] = customer.apply(segment, axis=1)
customer.head()

In [ ]:
customer.groupby('customer_segment').agg(customers=('customer_id','count'), avg_orders=('orders','mean'), avg_profit=('total_profit','mean'), avg_margin=('avg_margin','mean'), delay_rate=('delay_rate','mean'), avg_rating=('avg_rating','mean')).sort_values('avg_profit', ascending=False)